# Google Colab Remote Execution Environment

Use this notebook connected to a Colab T4 GPU (or better) when running computationally expensive models and tuning. This is an operational runbook rather than a canonical analysis notebook, so it is allowed to stay focused on setup, transfers, and remote execution steps.

## Set up Colab environment

Mount to Google Drive then choose a remote path to load the repo into and the cloud Drive path where the necessary data artifact files are (to prevent the need for preprocessing runs).

In [1]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive', force_remount=True)

REPO_URL = "https://github.com/DavisM1212/NHTSA-ODI-Complaint-Analytics.git"
REPO_DIR = Path("/content/NHTSA-ODI-Complaint-Analytics")
DRIVE_ROOT = Path("/content/drive/MyDrive/nhtsa_odi_colab")

Mounted at /content/drive


Clone repo if it doesn't exist and set it as the working directory

In [2]:
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

/content/NHTSA-ODI-Complaint-Analytics


Install dependencies. This can't be done through the requirements.txt since Colab uses Python 3.12.13 and the repo is set for 3.13.12, so the dependencies are different.

In [3]:
%pip install --upgrade pip setuptools wheel
%pip install --no-cache-dir \
  numpy \
  pandas \
  pyarrow \
  scikit-learn \
  catboost==1.2.10 \
  optuna


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 50.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.3 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 63.3 MB/s  0:00:01 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [catboost]2/3 [catboost]


## Model Runs

Make sure the model is run on GPU with devices set to 0 to select the correct one. Add any other needed arguments and set a script to copy outputs back over to the cloud Drive when done. This can take a long time to run even on power GPUs, 3+ hours for the most comprehensive modeling runs. If you run it overnight without copying the files over (or if there's a bug in how you set the copy script) you run the risk of the remote Colab Server idling too long and deleting itself along with your outputs.

### Component Feature Wave 1 Model

Copy over data artifacts needed for modeling runs

In [ ]:
!mkdir -p data/processed data/outputs
!cp {DRIVE_ROOT/'odi_component_single_label_cases.parquet'} data/processed/
!cp {DRIVE_ROOT/'odi_component_multilabel_cases.parquet'} data/processed/
!cp {DRIVE_ROOT/'component_single_label_benchmark_manifest.json'} data/outputs/
!cp {DRIVE_ROOT/'component_single_label_selection_manifest.json'} data/outputs/
!cp {DRIVE_ROOT/'component_single_label_holdout_calibration.csv'} data/outputs/
!cp {DRIVE_ROOT/'component_multilabel_manifest.json'} data/outputs/

In [ ]:
!cd notebooks/archive &&
!python -m component_feature_wave1 --task-type GPU --devices 0 --skip-single

!cp data/outputs/'component_featurewave1_manifest.json' {DRIVE_ROOT/'component_featurewave1_manifest_multi.json'}

[run] Multi-label structured feature wave 1
[multi] Split rows | train_core=205,913 screen_2024=62,449 select_2025=68,036 holdout_2026=10,192
[multi] screen_2024 1/5 -> core_structured
[multi] screen_2024 core_structured done macro_f1=0.1986 micro_f1=0.4239 recall@3=0.6458
[multi] screen_2024 2/5 -> wave1_incident_bundle
[multi] screen_2024 wave1_incident_bundle done macro_f1=0.1978 micro_f1=0.4221 recall@3=0.6423
[multi] screen_2024 3/5 -> wave1_date_quality_bundle
[multi] screen_2024 wave1_date_quality_bundle done macro_f1=0.1986 micro_f1=0.4249 recall@3=0.6460
[multi] screen_2024 4/5 -> wave1_geo_time_bundle
[multi] screen_2024 wave1_geo_time_bundle done macro_f1=0.1919 micro_f1=0.4070 recall@3=0.6369
[multi] screen_2024 5/5 -> wave1_cohort_history_bundle
[multi] screen_2024 wave1_cohort_history_bundle done macro_f1=0.1951 micro_f1=0.4009 recall@3=0.6424
[multi] select_2025 1/7 -> core_structured
[multi] select_2025 core_structured done fit=174.65s iter=1475 threshold=0.175 macro_f1

## Component Text Wave 2 Model

In [ ]:
!mkdir -p data/processed data/outputs
!cp {DRIVE_ROOT/'odi_component_single_label_cases.parquet'} data/processed/
!cp {DRIVE_ROOT/'odi_component_multilabel_cases.parquet'} data/processed/
!cp {DRIVE_ROOT/'odi_component_text_sidecar.parquet'} data/processed/
!cp {DRIVE_ROOT/'component_single_label_benchmark_manifest.json'} data/outputs/
!cp {DRIVE_ROOT/'component_single_label_benchmark_metrics.csv'} data/outputs/
!cp {DRIVE_ROOT/'component_single_label_selection_manifest.json'} data/outputs/
!cp {DRIVE_ROOT/'component_single_label_holdout_calibration.csv'} data/outputs/
!cp {DRIVE_ROOT/'component_multilabel_manifest.json'} data/outputs/
!cp {DRIVE_ROOT/'component_multilabel_metrics.csv'} data/outputs/

In [ ]:
!cd notebooks/archive &&
!python -m component_text_wave2 --task-type GPU --devices 0 --skip-text-plus --final-linear-model sgd

[run] Single-label text wave 2
[single] Split rows | train_core=151,075 screen_2024=45,584 select_2025=48,013 holdout_2026=6,995 final_linear_model=sgd
[single] screen_2024 structured_carry_forward macro_f1=0.2721 top1=0.4260 top3=0.6666
[single] screen_2024 text_only_linear macro_f1=0.7193 top1=0.8450 top3=0.9526
[single] screen_2024 text_plus_structured_linear skipped by flag
[single] screen_2024 text_structured_late_fusion macro_f1=0.7211 top1=0.8507 top3=0.9454
[single] select_2025 structured_carry_forward macro_f1=0.2711 top1=0.4094 top3=0.6516
[single] select_2025 text_only_linear macro_f1=0.7134 top1=0.8430 top3=0.9543
[single] select_2025 text_plus_structured_linear skipped by flag
[single] select_2025 text_structured_late_fusion macro_f1=0.7151 top1=0.8466 top3=0.9455
[single] select_2025 best=text_structured_late_fusion macro_f1=0.7151 delta_vs_locked=+0.4469 gate_pass=true
[single] holdout_2026 start family=text_structured_late_fusion
[single] holdout_2026 fitting late-fusio

## Component Text Wave 2b Calibration

In [3]:
!mkdir -p data/processed data/outputs
!cp {DRIVE_ROOT/'odi_component_single_label_cases.parquet'} data/processed/
!cp {DRIVE_ROOT/'odi_component_multilabel_cases.parquet'} data/processed/
!cp {DRIVE_ROOT/'odi_component_text_sidecar.parquet'} data/processed/
!cp {DRIVE_ROOT/'component_single_label_benchmark_manifest.json'} data/outputs/
!cp {DRIVE_ROOT/'component_single_label_benchmark_metrics.csv'} data/outputs/
!cp {DRIVE_ROOT/'component_single_label_selection_manifest.json'} data/outputs/
!cp {DRIVE_ROOT/'component_single_label_holdout_calibration.csv'} data/outputs/
!cp {DRIVE_ROOT/'component_multilabel_manifest.json'} data/outputs/
!cp {DRIVE_ROOT/'component_multilabel_metrics.csv'} data/outputs/
!cp {DRIVE_ROOT/'component_textwave2_manifest.json'} data/outputs/

In [4]:
!python -m src.modeling.official.component_single_text_calibrated --task-type GPU --devices 0 --final-linear-model sgd --alpha-grid 1,1.25,1.5,1.75,2,2.5,3,4,5,6,8

[wave2b] rows | dev_screen=196,659 select_2025=48,013 dev_select=244,672 holdout_2026=6,995
[wave2b] family=text_structured_late_fusion text_weight=0.75 final_linear_model=sgd
[wave2b] fitting select structured branch
[wave2b] select structured fit_seconds=172.43 iteration=900
[wave2b] fitting select text branch
[wave2b] select text fit_seconds=105.09
[wave2b] selected alpha=1.500 select_ece=0.0157
[wave2b] fitting holdout structured branch
[wave2b] holdout structured fit_seconds=103.36
[wave2b] fitting holdout text branch
[wave2b] holdout text fit_seconds=231.33
[wave2b] holdout calibrated macro_f1=0.7456 top1=0.8522 top3=0.9497 ece=0.0248 promotion_status=promoted
[write] /content/NHTSA-ODI-Complaint-Analytics/data/outputs/component_textwave2b_calibration_manifest.json
